In [1]:
from finlab.data import Data
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
#get 00878
import pandas as pd
import yfinance as yf
import pandas as pd

def getstkPrice(stkid,start,end):
    df = yf.download(stkid,start=start, end=end)
    #df.rename()
    file = f"{stkid}.pikle"
    df.to_pickle(file)
    return df

def getstkPricefromPickle(file):
    df = pd.read_pickle(file)
    #df.rename()
    return df


data = Data()
s89 = data.get("上月營收")
#s89.to_csv('allstock.csv')
df_revenue = s89['8299']

# 读取公司月營收数据
df_revenue = s89['8299']
#df_revenue['Date'] = pd.to_datetime(df_revenue['Date'])
#df_revenue = df_revenue.set_index('Date')
df_revenue = df_revenue.resample('M').mean()
df_revenue = df_revenue.dropna()
df_revenue



stkid = '8299.TWO'
divstkid = '8299'
file = f"{stkid}.pikle"

#df_stock = getstkPrice('8299.TWO','2001-01-01','2023-06-01')
#df_stock
#df.to_pickle(file)
df_stock =  getstkPricefromPickle(file)
df_stock = df_stock.reset_index().rename(columns={'Date':'date'})

df_stock['date'] = pd.to_datetime(df_stock['date'])
df_stock = df_stock.set_index('date')
df_stock = df_stock.resample('M').mean()
df_stock = df_stock.dropna()
df_stock=df_stock['Close']
df_stock



date
2007-12-31    167.630066
2008-01-31    154.190997
2008-02-29    133.714087
2008-03-31    146.008054
2008-04-30    188.153577
                 ...    
2023-01-31    350.346154
2023-02-28    362.583333
2023-03-31    361.934783
2023-04-30    397.588235
2023-05-31    390.613636
Freq: M, Name: Close, Length: 186, dtype: float64

In [ ]:
df_revenue

In [ ]:
df

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,


In [2]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# 组合数据集
df = pd.concat([df_revenue, df_stock], axis=1)
df = df.dropna()

# 归一化数据集
scaler = MinMaxScaler()
df_norm = scaler.fit_transform(df.values)

In [ ]:
df_norm

In [8]:
# 定义训练集和验证集
train_size = int(len(df_norm) * 0.6)
validation_size = int(len(df_norm) * 0.2)
#test_size = len(df_norm) - train_size - validation_size
test_size = len(df_norm) - train_size #- validation_size
train_set = df_norm[:train_size, :]
validation_set = df_norm[train_size:train_size+validation_size, :]
test_set = df_norm[train_size+validation_size:, :]

In [9]:
# 创建特征和目标
def create_dataset(dataset):
    x, y = [], []
    dx = 10
    #for i in range(len(dataset)-30-1):
    for i in range(len(dataset)-dx-1):
        a = dataset[i:(i+dx)]
        x.append(a)
        y.append(dataset[i+dx])
    return np.array(x), np.array(y)

In [10]:
# 创建训练集的特征和目标
x_train, y_train = create_dataset(train_set)

# 创建并编译LSTM模型
model = Sequential()
model.add(LSTM(units=50, return_sequences=True, input_shape=(x_train.shape[1], 2)))
model.add(LSTM(units=50))
model.add(Dense(units=1))
model.compile(optimizer='adam', loss='mean_squared_error')

# 训练LSTM模型
model.fit(x_train, y_train, epochs=100, batch_size=32)

Epoch 1/100
2/2 [==============================] - 2s 8ms/step - loss: 0.0696
Epoch 2/100
2/2 [==============================] - 0s 5ms/step - loss: 0.0370
Epoch 3/100
2/2 [==============================] - 0s 5ms/step - loss: 0.0194
Epoch 4/100
2/2 [==============================] - 0s 6ms/step - loss: 0.0188
Epoch 5/100
2/2 [==============================] - 0s 5ms/step - loss: 0.0241
Epoch 6/100
2/2 [==============================] - 0s 6ms/step - loss: 0.0217
Epoch 7/100
2/2 [==============================] - 0s 6ms/step - loss: 0.0175
Epoch 8/100
2/2 [==============================] - 0s 5ms/step - loss: 0.0160
Epoch 9/100
2/2 [==============================] - 0s 6ms/step - loss: 0.0173
Epoch 10/100
2/2 [==============================] - 0s 10ms/step - loss: 0.0189
Epoch 11/100
2/2 [==============================] - 0s 6ms/step - loss: 0.0196
Epoch 12/100
2/2 [==============================] - 0s 5ms/step - loss: 0.0191
Epoch 13/100
2/2 [==============================] - 0s 5ms/s

In [11]:
# 创建验证集的特征和目标
x_validation, y_validation = create_dataset(validation_set)

# 使用验证集评估模型


In [12]:
print(y_validation)
print(x_validation)
#len(validation_set)

[[0.75541857 0.63547285]
 [0.81656849 0.53158877]
 [0.82508123 0.62273476]
 [0.90656268 0.86960826]]
[[[0.44444566 0.35104174]
  [0.47353192 0.41528988]
  [0.4109703  0.54299754]
  [0.39912643 0.67235667]
  [0.3680338  0.83082429]
  [0.25735135 1.        ]
  [0.67104613 0.88842463]
  [0.64869434 0.87074907]
  [0.81370284 0.87990683]
  [0.65762568 0.78612412]]

 [[0.47353192 0.41528988]
  [0.4109703  0.54299754]
  [0.39912643 0.67235667]
  [0.3680338  0.83082429]
  [0.25735135 1.        ]
  [0.67104613 0.88842463]
  [0.64869434 0.87074907]
  [0.81370284 0.87990683]
  [0.65762568 0.78612412]
  [0.75541857 0.63547285]]

 [[0.4109703  0.54299754]
  [0.39912643 0.67235667]
  [0.3680338  0.83082429]
  [0.25735135 1.        ]
  [0.67104613 0.88842463]
  [0.64869434 0.87074907]
  [0.81370284 0.87990683]
  [0.65762568 0.78612412]
  [0.75541857 0.63547285]
  [0.81656849 0.53158877]]

 [[0.39912643 0.67235667]
  [0.3680338  0.83082429]
  [0.25735135 1.        ]
  [0.67104613 0.88842463]
  [0.6486

In [16]:
#loss, acc = model.evaluate(x_validation, y_validation)

loss = model.evaluate(x_validation, y_validation)
print('Validation accuracy:', loss)

1/1 [==============================] - 0s 17ms/step - loss: 0.0995
Validation accuracy: 0.09948712587356567


In [34]:
# 创建测试集的特征和目标
x_test, y_test = create_dataset(test_set)

# 使用测试集预测
predictions = model.predict(x_test)

# 将预测结果反归一化
predictions = scaler.inverse_transform(predictions)

# 将实际结果反归一化
y_test = scaler.inverse_transform(y_test.reshape(-1, 1))

# 检查预测结果
predictions_direction = []  # 存储预测的股票价格涨跌方向
for i in range(len(predictions) - 1):
    if predictions[i+1][0] > predictions[i][0]:
        predictions_direction.append(1)
    else:
        predictions_direction.append(0)

actual_direction = []  # 存储实际的股票价格涨跌方向
for i in range(len(y_test) - 1):
    if y_test[i+1][0] > y_test[i][0]:
        actual_direction.append(1)
    else:
        actual_direction.append(0)

# 计算预测结果的正确率
correct = 0
for i in range(len(predictions_direction)):
    if predictions_direction[i] == actual_direction[i]:
        correct += 1

accuracy = correct / len(predictions_direction)
print('Test accuracy:', accuracy)


1/1 [==============================] - 0s 62ms/step


ValueError: non-broadcastable output operand with shape (5,1) doesn't match the broadcast shape (5,2)

In [26]:
type(predictions)
predictions[0] = predictions[0].reshape(1,2)
predictions

ValueError: cannot reshape array of size 1 into shape (5,2)

In [33]:
aaa = predictions
aaa = aaa.reshape(5,2)
aaa

ValueError: cannot reshape array of size 5 into shape (5,2)